# Lab 2, Part 1: from S4 to S6 — what selection adds to a pretrained Mamba

Checkpoints (each gates the next stage):
- **CP1** baseline English ppl ≈ 10.56
- **CP2** V4 == stock logits (max abs diff < 1e-3, fp32, eval)
- **CP3** train-mode == eval-mode logits for every variant
- **CP4** kernel call counter: fused path in training, selective-scan in eval, no fallback
- **CP5** V4 after recovery not worse than baseline

Runs on one H100 (allowed for Part 1 only). Same numeric recipe as the T4 plan: fp32 weights, fp16 autocast + GradScaler.

## 0. Environment (Route B: mamba-ssm + causal-conv1d prebuilt wheels, torch ≤ 2.10)

In [1]:
import torch, sys
print(sys.version, torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
!nvcc --version | tail -2

3.13.16 (main, Oct  3 2026, 01:06:22) [Clang 22.1.3 ] 2.10.0+cu128 12.8 NVIDIA H100 NVL


/bin/bash: line 1: nvcc: command not found


In [2]:
# Route B. If the wheels do not match this torch, use Route A: pip install kernels, then the Hub kernels
# kernels-community/mamba-ssm with an explicit version (check kernels.has_kernel) and Mode.TRAINING.
!MAMBA_KEEP_CUDA_BUILD=TRUE pip install -q mamba-ssm==2.3.2.post1 --no-build-isolation
!pip install -q causal-conv1d==1.7.0 --no-build-isolation
!pip install -q -e ..

In [3]:
import importlib.metadata as im, json, os
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, MambaForCausalLM
from lab2.common import load_config, out_dir
from lab2 import wandb_log as wl
from lab2.part1.variants import KERNELS, VARIANTS, apply_variant, Calibrator, VariantChecks
from lab2.part1.data import EvalSamples, TrainStream, overlap
from lab2.part1.evaluate import KernelCounter, LMEvaluator
from lab2.part1.recovery import RecoveryTrainer, param_counts, lr_factor

assert KERNELS, "mamba_ssm / causal_conv1d not importable: training would fall back to the slow path"
cfg = load_config()
p1, SEED, DEV = cfg["part1"], cfg["seed"], "cuda"
OUT = out_dir(cfg, "part1")                 # large outputs (samples, stream, checkpoints)
FIG = Path("../report/figures/part1")       # small tables / figures for the report
FIG.mkdir(parents=True, exist_ok=True)
ENV = {"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
       **{k: im.version(k) for k in ("transformers", "mamba_ssm", "causal_conv1d")}}
print(ENV)
# os.environ["WANDB_API_KEY"] = "..."   # W&B is disabled without a key; everything is also saved locally

RESULTS = OUT / "results.json"
results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
def save_result(key, value):
    results[key] = value
    RESULTS.write_text(json.dumps(results, indent=1))

tok = AutoTokenizer.from_pretrained(p1["model"])
def load_stock():
    return MambaForCausalLM.from_pretrained(p1["model"], dtype=torch.float32).to(DEV)

/home/pkhomenko/ai500-labs/lab-2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'torch': '2.10.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA H100 NVL', 'transformers': '5.23.0', 'mamba_ssm': '2.3.2.post1', 'causal_conv1d': '1.7.0'}


## 1. Fixed evaluation samples and the training stream (built once, then only loaded)

In [4]:
samples_path = OUT / "eval_samples.pt"
es = EvalSamples(tok, p1["seq_len"])
if not samples_path.exists():
    torch.save(es.build_all(p1["data"], SEED), samples_path)
samples = torch.load(samples_path)
for name, s in samples.items():
    print(name, "docs", len(s["ids"]), "tokens", s["tokens"], "bytes", int(s["bytes"].sum()))
assert len(samples["en"]["ids"]) >= p1["data"]["min_docs"] and len(samples["uk"]["ids"]) >= p1["data"]["min_docs"]

# reproducibility: rebuild the (small) val sample with the same seed -> identical ids
pile = f"hf://datasets/{p1['data']['pile']}"
from lab2.part1.data import stream_texts
val_again = es.build(stream_texts(f"{pile}/{p1['data']['val_file']}", SEED, p1["data"]["shuffle_buffer"]), p1["data"]["val_tokens"])
assert EvalSamples.same(samples["val"], val_again)
short = sum(len(x) < p1["seq_len"] for x in samples["en"]["ids"])
print("en docs shorter than seq_len (kept whole):", short)

en docs 2788 tokens 2001067 bytes 8180724
uk docs 1921 tokens 2000559 bytes 6066528
val docs 294 tokens 200128 bytes 824896


en docs shorter than seq_len (kept whole): 2381


In [5]:
stream_dir = out_dir(cfg, "part1", "stream")
if not (stream_dir / "train.npy").exists():
    print(TrainStream(tok, p1["seq_len"]).build(p1["data"], SEED, stream_dir))
calib_rows, train_rows = TrainStream.load(stream_dir)
print("calib rows", tuple(calib_rows.shape), "train rows", tuple(train_rows.shape))
ov = overlap((stream_dir / "train_keys.txt").read_text().split(), samples)
print("eval docs found in calib/train stream:", ov)
assert not any(ov.values())

calib rows (488, 2048) train rows (14648, 2048)
eval docs found in calib/train stream: {'en': 0, 'uk': 0, 'val': 0}


## 2. Baseline (V4 zero-shot = unmodified model) — CP1, CP4

In [6]:
wl.init_run(cfg, "baseline", "eval", {**p1, "env": ENV, "seed": SEED})
model = load_stock()
# CP4 on the training path: one fp16-autocast step must hit the fused kernel
counter = KernelCounter()
with counter():
    model.train()
    with torch.autocast("cuda", dtype=torch.float16):
        model(input_ids=train_rows[:2].to(DEV), labels=train_rows[:2].to(DEV), use_cache=False).loss.backward()
print("train step:", counter.assert_path("fused"))
model.zero_grad(set_to_none=True)

ev = LMEvaluator(DEV)
base = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
print(json.dumps(base, indent=1))
save_result("zeroshot/V4", base)
wl.log_summary({"en/ppl": base["en"]["ppl"], "en/bpb": base["en"]["bpb"], "uk/bpb": base["uk"]["bpb"],
                "published_ppl": p1["published_ppl"], **param_counts(model)})
wl.finish()
print(f"CP1: ppl {base['en']['ppl']:.3f} vs published {p1['published_ppl']} (test vs validation split)")

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.


wandb: Currently logged in as: pavlokhomenko2307 (ai340-nlp) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_161958-mobaoew0
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run baseline


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


... (9 outputs left out)

wandb: 
wandb: Run summary:
wandb:           en/bpb 0.80368
wandb:           en/ppl 9.75133
wandb:     params_total 129135360
wandb: params_trainable 129135360
wandb:    published_ppl 10.56
wandb:           uk/bpb 1.05621
wandb: 


wandb: 🚀 View run baseline at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/mobaoew0
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_161958-mobaoew0/logs


CP1: ppl 9.751 vs published 10.56 (test vs validation split)


## 3. Calibration of β, B, C on ~1M train tokens (stock model, eval mode)

In [7]:
calib_path = OUT / "calibration.pt"
wl.init_run(cfg, "calibration", "calibrate", {"tokens": int(calib_rows.numel())})
if not calib_path.exists():
    Calibrator.save(Calibrator(load_stock()).run(calib_rows), calib_path)
calib = Calibrator.load(calib_path)
ratios = pd.DataFrame({"layer": range(len(calib["ratio_B"])), "ratio_B": calib["ratio_B"].numpy(),
                       "ratio_C": calib["ratio_C"].numpy()})
wl.log_table("calibration_ratio", ratios, FIG / "calibration_ratio")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ratios.layer, ratios.ratio_B, marker="o", label="B")
ax.plot(ratios.layer, ratios.ratio_C, marker="o", label="C")
ax.set(xlabel="layer", ylabel="‖mean‖ / mean‖·‖", title="How much of B_k, C_k survives averaging")
ax.legend()
wl.log_figure("calibration_ratio_plot", fig, FIG / "calibration_ratio.png")
wl.finish()
ratios

wandb: setting up run 6myl5kf7


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_162122-6myl5kf7
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run calibration


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/6myl5kf7


wandb: updating run metadata; uploading artifact run-6myl5kf7-calibration_ratio; uploading summary


wandb: uploading artifact run-6myl5kf7-calibration_ratio


wandb: 🚀 View run calibration at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/6myl5kf7
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 2 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_162122-6myl5kf7/logs


,layer,ratio_B,ratio_C
0,0,0.946383,0.838383
1,1,0.798348,0.786894
2,2,0.770770,0.751295
3,3,0.887565,0.863258
4,4,0.899077,0.811259
5,5,0.886730,0.673784
6,6,0.891784,0.712621
7,7,0.926529,0.763875
8,8,0.885705,0.618500
9,9,0.850104,0.714933


## 4. Variant checks — CP2, CP3, static invariance, gradient reproducibility

In [8]:
wl.init_run(cfg, "checks", "test")
build = lambda v: apply_variant(load_stock(), v, calib)
batch = train_rows[:2, :1024].to(DEV)                      # unpadded
checks = VariantChecks(build, load_stock(), batch, p1["checks"]["v4_tol"], p1["checks"]["grad_rtol"])
counter = KernelCounter()
with counter():
    check_rows = checks.run_checks()
checks_df = pd.DataFrame(check_rows)
wl.log_table("checks", checks_df, FIG / "checks")
wl.log_summary({"kernel_counts": dict(counter.counts)})
wl.finish()
checks_df

wandb: setting up run ij9nvnns


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_162125-ij9nvnns
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run checks


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/ij9nvnns


... (34 outputs left out)

wandb: uploading history steps 0-0, summary


wandb: 🚀 View run checks at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/ij9nvnns
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_162125-ij9nvnns/logs


,check,variant,v4_vs_stock_max_abs,train_vs_eval_max_abs,train_vs_eval_rel,dt_diff_max,B_diff_max,C_diff_max,grad_rel_diff_max
0,V4 vs stock,V4,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,train vs eval,V1,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,train vs eval,V2,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,train vs eval,V3,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,train vs eval,V4,NaN,0.0,0.0,NaN,NaN,NaN,NaN
5,static invariance,V1,NaN,NaN,NaN,0.000000,0.000000,0.0,NaN
6,grad reproducibility,V1,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
7,static invariance,V2,NaN,NaN,NaN,88.329361,0.000000,0.0,NaN
8,grad reproducibility,V2,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
9,static invariance,V3,NaN,NaN,NaN,0.000000,40.398518,0.0,NaN


## 5. Zero-shot surgery

**Pre-registration:** write the predicted V1–V4 ordering by zero-shot English perplexity into
`report/part1.md` and commit it *before* running the next cell.

In [9]:
for v in ("V1", "V2", "V3"):
    if f"zeroshot/{v}" in results:
        continue
    wl.init_run(cfg, f"zeroshot-{v}", "eval", {"variant": v})
    model = build(v)
    r = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
    save_result(f"zeroshot/{v}", r)
    wl.log_summary({"en/ppl": r["en"]["ppl"], "en/bpb": r["en"]["bpb"], "uk/bpb": r["uk"]["bpb"], **param_counts(model)})
    wl.finish()
pd.DataFrame({v: {"en_ppl": results[f"zeroshot/{v}"]["en"]["ppl"], "table7": p1["table7_ppl"][v]} for v in VARIANTS}).T

wandb: setting up run 02v889wk


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_162139-02v889wk
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run zeroshot-V1


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/02v889wk


... (33 outputs left out)

wandb: 
wandb: Run summary:
wandb:           en/bpb 1.94911
wandb:           en/ppl 250.43921
wandb:     params_total 129725184
wandb: params_trainable 127955712
wandb:           uk/bpb 3.65347
wandb: 


wandb: 🚀 View run zeroshot-V3 at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/ayi3k4fc
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_162338-ayi3k4fc/logs


,en_ppl,table7
V1,427.898928,10.93
V2,44.750559,9.81
V3,250.439209,10.15
V4,9.751325,8.71


## 6. Recovery training (same rows, order, budget, seed, hyperparameters) — CP4, CP5

In [10]:
trainer = RecoveryTrainer(cfg, ev, samples["val"])

def recover(v, seed, max_steps=None):
    key = f"recovery/{v}/s{seed}"
    if key in results and max_steps is None:
        return results[key]
    run_dir = out_dir(cfg, "part1", "runs", f"{v}-s{seed}")
    wl.init_run(cfg, f"recovery-{v}-s{seed}", "train", {**p1["train"], "variant": v, "seed": seed, "env": ENV},
                group=f"recovery-s{seed}")
    model = build(v)
    history, summary = trainer.train(model, train_rows, seed, run_dir, max_steps)
    r = {name: ev.evaluate(model, samples[name]) for name in ("en", "uk")}
    wl.log_summary({"en/ppl": r["en"]["ppl"], "en/bpb": r["en"]["bpb"], "uk/bpb": r["uk"]["bpb"]})
    wl.finish()
    out = {**r, "history": history, "summary": summary}
    if max_steps is None:
        save_result(key, out)
    return out

# timing / sanity: ~20 steps of V4 (fit micro_batch in config.yaml if it does not fit)
t = recover("V4", SEED, max_steps=20)
print(t["summary"])

wandb: setting up run gd20n874


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_162436-gd20n874
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run recovery-V4-s42


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/gd20n874


... (4 outputs left out)

wandb: 
wandb: Run history:
wandb:             tokens ▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:    train/grad_norm ▃▅█▅▂▃▅▂▅▂▂▂▃▂▁▁▂▃▁▁
wandb:         train/loss ▆▄▆▅█▂▁▅▆▄▅▂▆▂▆▄▄█▃▂
wandb:           train/lr ▁▂▃▄▅▅▆▇████████████
wandb:        train/scale ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/step_seconds █▂▁▁▃▂▄▄▃▂▃▃▅▃▄▅▄▃▅▂
wandb:           val/loss █▁
wandb: 
wandb: Run summary:
wandb:           en/bpb 0.80364
wandb:           en/ppl 9.75015
wandb:     params_total 129135360
wandb: params_trainable 129135360
wandb:     step_seconds 1.02653
wandb:            steps 20
wandb:           tokens 1310720
wandb:  train/grad_norm 0.43538
wandb:       train/loss 2.07834
wandb:         train/lr 1e-05
wandb:               +5 ...
wandb: 


wandb: 🚀 View run recovery-V4-s42 at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/gd20n874
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_162436-gd20n874/logs


{'steps': 20, 'tokens': 1310720, 'wall_seconds': 20.530652046203613, 'step_seconds': 1.0265326023101806, 'kernel_counts': {'fused': 1920, 'conv': 7056, 'scan': 7056}, 'params_total': 129135360, 'params_trainable': 129135360}


In [11]:
v4 = recover("V4", SEED)
print("CP5:", v4["en"]["ppl"], "<=", base["en"]["ppl"])
assert v4["en"]["ppl"] <= base["en"]["ppl"], "V4 got worse with recovery: lower the lr and redo before V1-V3"

CP5: 9.750113730376976 <= 9.75132507671179


In [12]:
for v in ("V1", "V2", "V3"):
    recover(v, SEED)

wandb: setting up run pojbaqjx


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_162625-pojbaqjx
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run recovery-V1-s42


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/pojbaqjx


... (29 outputs left out)

wandb: uploading summary


wandb: 
wandb: Run history:
wandb:             tokens ▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇███
wandb:    train/grad_norm █▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         train/loss █▇▆▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▃▂▂▂▁▂▂▂▂▁▁▂▂▂▂▁▁▂▂▂▂▁
wandb:           train/lr ▃████▇▇▇▆▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        train/scale █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/step_seconds ▄▃▄▃█▁▅▅▅▄▂▁▃▇▃▃▁▃▃▂▁▅▂▄▃▆▂▂▂▃▂▄▅▃▂▃▂▄▃▄
wandb:           val/loss █▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:           en/bpb 1.41414
wandb:           en/ppl 54.99593
wandb:     params_total 129725184
wandb: params_trainable 127955712
wandb:     step_seconds 0.84526
wandb:            steps 457
wandb:           tokens 29949952
wandb:  train/grad_norm 1.29377
wandb:       train/loss 3.98081
wandb:         train/lr 0.0
wandb:               +5 ...
wandb: 


wandb: 🚀 View run recovery-V3-s42 at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/5385qmkm
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_164035-5385qmkm/logs


In [13]:
# second seed for the two variants closest in English ppl after recovery
ppl = {v: results[f"recovery/{v}/s{SEED}"]["en"]["ppl"] for v in VARIANTS}
ordered = sorted(ppl, key=ppl.get)
pair = min(zip(ordered, ordered[1:]), key=lambda ab: abs(ppl[ab[0]] - ppl[ab[1]]))
seed2 = p1["train"]["seeds"][1]
print("closest pair:", pair, "-> seed", seed2)
for v in pair:
    recover(v, seed2)

closest pair: ('V4', 'V2') -> seed 43


wandb: setting up run 53jeeduo


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_164815-53jeeduo
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run recovery-V4-s43


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


... (20 outputs left out)

wandb: uploading summary


wandb: 
wandb: Run history:
wandb:             tokens ▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb:    train/grad_norm ██▇▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▂▂▁▂▁▁▁▁▁▁▁
wandb:         train/loss ██▇▇▆▅▅▄▅▅▆▅▅▃▃▄▃▃▂▄▃▄▂▃▂▃▂▄▄▂▁▃▃▄▄▅▂▃▃▃
wandb:           train/lr ▃████████▇▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
wandb:        train/scale ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/step_seconds ▃▁▃▂▃▃▄▅▅▃▂█▃▂▃▂▂▂▁▃▂▃▃▃▃▅▃▂▃▁▃▁▂▂▂▁▃▂▂▂
wandb:           val/loss █▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:           en/bpb 1.09561
wandb:           en/ppl 22.30137
wandb:     params_total 130315008
wandb: params_trainable 130315008
wandb:     step_seconds 0.755
wandb:            steps 457
wandb:           tokens 29949952
wandb:  train/grad_norm 0.8621
wandb:       train/loss 3.13088
wandb:         train/lr 0.0
wandb:               +5 ...
wandb: 


wandb: 🚀 View run recovery-V2-s43 at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/esmgkbav
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_165626-esmgkbav/logs


## 7. Report tables and figures

In [14]:
rows = []
for v in VARIANTS:
    z, r = results[f"zeroshot/{v}"], results[f"recovery/{v}/s{SEED}"]
    row = {"variant": v, "table7_ppl": p1["table7_ppl"][v],
           "zs_en_ppl": z["en"]["ppl"], "zs_en_bpb": z["en"]["bpb"], "zs_uk_bpb": z["uk"]["bpb"],
           "rec_en_ppl": r["en"]["ppl"], "rec_en_bpb": r["en"]["bpb"], "rec_uk_bpb": r["uk"]["bpb"],
           "params_trainable": r["summary"]["params_trainable"], "train_hours": r["summary"]["wall_seconds"] / 3600,
           "step_seconds": r["summary"]["step_seconds"]}
    s2 = results.get(f"recovery/{v}/s{seed2}")
    if s2:
        row.update({"rec_en_ppl_s2": s2["en"]["ppl"], "rec_uk_bpb_s2": s2["uk"]["bpb"]})
    rows.append(row)
table = pd.DataFrame(rows)
table.loc[table.variant == "V4", "published_ppl"] = p1["published_ppl"]
wl.init_run(cfg, "report", "report")
wl.log_table("results", table, FIG / "results")

fig, ax = plt.subplots(figsize=(8, 4.5))
for v in VARIANTS:
    h = results[f"recovery/{v}/s{SEED}"]["history"]
    ax.plot([p["tokens"] / 1e6 for p in h], [p["val_loss"] for p in h], marker=".", label=v)
ax.set(xlabel="training tokens (M)", ylabel="val loss (200k-token sample)", title="Recovery curves", yscale="log")
ax.legend()
wl.log_figure("recovery_curves", fig, FIG / "recovery_curves.png")
kernels = pd.DataFrame([{"run": k, **v["summary"]["kernel_counts"]} for k, v in results.items() if k.startswith("recovery")])
wl.log_table("kernel_counts", kernels, FIG / "kernel_counts")
wl.finish()
table

wandb: setting up run vwq76scf


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /home/pkhomenko/ai500-labs/lab-2/notebooks/wandb/run-20261116_170324-vwq76scf
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run report


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1


wandb: 🚀 View run at https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/vwq76scf


... (3 outputs left out)

wandb: uploading history steps 0-2, summary


wandb: 🚀 View run report at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1/runs/vwq76scf
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/ai340-nlp/ai500-lab2-part1
wandb: Synced 4 W&B file(s), 3 media file(s), 4 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261116_170324-vwq76scf/logs


,variant,table7_ppl,zs_en_ppl,zs_en_bpb,zs_uk_bpb,rec_en_ppl,rec_en_bpb,rec_uk_bpb,params_trainable,train_hours,step_seconds,rec_en_ppl_s2,rec_uk_bpb_s2,published_ppl
0,V1,10.93,427.898928,2.138146,4.225117,70.795729,1.503258,2.567865,128545536,0.100409,0.790970,NaN,NaN,NaN
1,V2,9.81,44.750559,1.341387,2.114315,22.188924,1.093828,1.918335,130315008,0.095197,0.749910,22.301365,1.919395,NaN
2,V3,10.15,250.439209,1.949111,3.653468,54.995935,1.414138,2.382668,127955712,0.107301,0.845262,NaN,NaN,NaN
3,V4,8.71,9.751325,0.803682,1.056208,9.750114,0.803638,1.056210,129135360,0.117668,0.926922,9.750167,1.056365,10.56
